# 3. Machine Learning for Classification

Logistic regression to predict churn

## 3.1 Churn Prediction project 
- Dataset: https://www.kaggle.com/blastchar/telco-customer-churn

## 3.2 Data Preparation

- Download the data, read it with pandas
- Look at the data
- Make column names and values look uniform
- Check if all the columns read correctly
- Check if the churn variable needs any preparation


In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [6]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [7]:
!wget $data -O data-week-3.csv 

--2026-01-16 19:38:31--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.009s  

2026-01-16 19:38:32 (103 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [8]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
df.shape

(7043, 21)

As we cannot see all the columns contents which are replaced by ... we can do so by taking transpose of head of matrix

In [10]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [11]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [12]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [13]:
df.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

In [14]:
pd.to_numeric(df['totalcharges'])

ValueError: Unable to parse string "_" at position 488

The above error indicates that total_charges does not only contain numbers to be parsed to numeric but also string "_" because in the previous cell we replaced " " with _

In [15]:
tc = pd.to_numeric(df['totalcharges'], errors='coerce')

In [16]:
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')

In [17]:
df['totalcharges'] = df['totalcharges'].fillna(0)

In [18]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,0.0
753,3115-czmzd,0.0
936,5709-lvoeq,0.0
1082,4367-nuyao,0.0
1340,1371-dwpaz,0.0
3331,7644-omvmy,0.0
3826,3213-vvolg,0.0
4380,2520-sgtta,0.0
5218,2923-arzlg,0.0
6670,4075-wkniu,0.0


In [19]:
df['churn'] = (df['churn'] == 'yes').astype(int)

In [20]:
df.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,7003,7004,7005,7006,7007,7008,7009,7010,7011,7012,7013,7014,7015,7016,7017,7018,7019,7020,7021,7022,7023,7024,7025,7026,7027,7028,7029,7030,7031,7032,7033,7034,7035,7036,7037,7038,7039,7040,7041,7042
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu,9305-cdskc,1452-kiovk,6713-okomc,7892-pookp,6388-tabgu,9763-grskd,7469-lkbci,8091-ttvax,0280-xjgex,5129-jlpis,3655-snqyz,8191-xwszg,9959-wofkt,4190-mfluw,4183-myfrb,8779-qrdmv,1680-vdcww,1066-jksgk,3638-weabw,6322-hrpfa,6865-jznko,6467-chfzw,8665-utdhz,5248-ygijn,8773-hhuoz,3841-nfecx,4929-xihvw,6827-ieauq,7310-egvhz,3413-bmnze,6234-raapl,6047-yhpvi,6572-adkrs,5380-wjkov,8168-uqwwf,...,4501-vcpfk,6075-slnil,9347-aerrl,0093-xwzfy,2274-xuata,1980-kxvpm,7703-zekef,0723-drclg,5482-nupna,6691-cciha,1685-bqula,9053-ejunl,0666-uxtjo,1471-giqkq,4807-izyoz,1122-jwtjw,9710-njern,9837-fwlch,1699-hpsbg,7203-oykct,1035-ipqpu,7398-lxgyx,2823-lkabh,8775-cebbj,0550-dcxlh,9281-cedru,2235-dwlju,0871-opbxw,3605-jiskb,6894-lfhly,9767-fflem,0639-tsiqw,8456-qdavc,7750-eyxwz,2569-wgero,6840-resvb,2234-xaduh,4801-jzazl,8361-ltmkd,3186-ajiek
gender,female,male,male,male,female,female,male,female,female,male,male,male,male,male,male,female,female,male,female,female,male,male,male,female,male,female,male,male,male,female,female,male,female,male,male,female,male,female,male,female,...,male,male,male,male,male,female,male,female,female,female,female,male,male,female,female,male,female,male,male,male,female,male,female,female,male,female,female,female,male,male,male,female,male,female,female,male,female,female,male,male
seniorcitizen,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,...,0,0,0,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,1,0
partner,yes,no,no,no,no,no,no,no,yes,no,yes,no,yes,no,no,yes,no,no,yes,no,no,yes,no,yes,yes,no,yes,yes,yes,no,yes,yes,yes,no,no,yes,no,no,no,no,...,no,no,yes,no,yes,no,no,yes,no,yes,no,no,yes,no,no,yes,no,yes,no,no,yes,yes,no,no,no,yes,no,no,yes,no,no,no,no,no,no,yes,yes,yes,yes,no
dependents,no,no,no,no,no,no,yes,no,no,yes,yes,no,no,no,no,yes,no,yes,yes,no,no,no,no,no,yes,no,yes,yes,no,yes,no,no,yes,no,no,yes,no,no,no,no,...,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,no,yes,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,no,yes,yes,yes,no,no
tenure,1,34,2,45,2,8,22,10,28,62,13,16,58,49,25,69,52,71,10,21,1,12,1,58,49,30,47,1,72,17,71,2,27,1,1,72,5,46,34,11,...,26,38,23,40,72,3,23,1,4,62,40,41,34,1,51,1,39,12,12,72,63,44,18,9,13,68,6,2,55,1,38,67,19,12,72,24,72,11,4,66
phoneservice,no,yes,yes,no,yes,yes,yes,no,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,no,yes,yes,yes,yes,yes,yes,no,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,...,no,yes,yes,yes,no,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,no,yes,yes,yes,yes,yes,yes,no,yes,yes,yes,no,yes,yes
multiplelines,no_phone_service,no,no,no_phone_service,no,yes,yes,no_phone_service,yes,no,no,no,yes,yes,no,yes,no,yes,no,no,no_phone_service,no,no,yes,no,no,yes,no_phone_service,yes,no,yes,no,no,no,no,yes,no,no,yes,yes,...,no_phone_service,yes,no,yes,no_phone_service,yes,yes,yes,no,yes,yes,yes,no,no,no,no,no,no,no,yes,yes,yes,yes,no,no,no,no_phone_service,no,yes,yes,no,yes,no,no_phone_service,no,yes,yes,no_phone_service,yes,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic,fiber_optic,fiber_optic,dsl,fiber_optic,dsl,dsl,no,fiber_optic,fiber_optic,fiber_optic,fiber_optic,no,fiber_optic,dsl,fiber_optic,dsl,no,no,dsl,dsl,dsl,fiber_optic,dsl,dsl,dsl,fiber_optic,fiber_optic,dsl,no,dsl,fiber_optic,fiber_optic,fiber_optic,fiber_optic,fiber_optic,...,dsl,fiber_optic,no,fiber_optic,dsl,fiber_optic,fiber_optic,fiber_optic,dsl,dsl,fiber_optic,fiber_optic,fiber_optic,dsl,no,fiber_optic,no,no,dsl,fiber_optic,fiber_optic,fiber_optic,fiber_optic,dsl,dsl,dsl,dsl,no,dsl,fiber_optic,fiber_optic,fiber_optic,fiber_optic,dsl,n

## 3.3 Setting up Validation Framework

- Perform train/test/validation split using Sklearn package

In [21]:
from sklearn.model_selection import train_test_split

In [22]:
#to get documentation of package
train_test_split?

Signature:
train_test_split(
    *arrays,
    test_size=None,
    train_size=None,
    random_state=None,
    shuffle=True,
    stratify=None,
)
Docstring:
Split arrays or matrices into random train and test subsets.

Quick utility that wraps input validation,
``next(ShuffleSplit().split(X, y))``, and application to input data
into a single call for splitting (and optionally subsampling) data into a
one-liner.

Read more in the :ref:`User Guide <cross_validation>`.

Parameters
----------
*arrays : sequence of indexables with same length / shape[0]
    Allowed inputs are lists, numpy arrays, scipy-sparse
    matrices or pandas dataframes.

test_size : float or int, default=None
    If float, should be between 0.0 and 1.0 and represent the proportion
    of the dataset to include in the test split. If int, represents the
    absolute number of test samples. If None, the value is set to the
    complement of the train size. If ``train_size`` is also None, it will
    be set to 0.25.

trai

In [25]:
df_fulltrain, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [26]:
df_train, df_val = train_test_split(df_fulltrain, test_size=0.25, random_state=1)

In [29]:
len(df_fulltrain), len(df_train), len(df_val), len(df_test)

(5634, 4225, 1409, 1409)

In [32]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [33]:
y_train = df_train['churn'].values
y_val = df_val['churn'].values
y_test = df_test['churn'].values

In [34]:
del df_train['churn']
del df_val['churn']
del df_test['churn']


## 3.4 EDA

- check missing values
- look at target vriable (churn)
- look at numerical and categorical variables

In [35]:
df_fulltrain = df_fulltrain.reset_index(drop=True)

In [36]:
df_fulltrain.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [37]:
df_fulltrain['churn'].value_counts()

churn
0    4113
1    1521
Name: count, dtype: int64

In [38]:
# to see percentage

df_fulltrain['churn'].value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

27% is our global churn rate which is same as mean of churn column

In [41]:
global_churn_rate = df_fulltrain['churn'].mean()
round(global_churn_rate, 2)

np.float64(0.27)

In [42]:
df_fulltrain.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

In [43]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

In [44]:
df_fulltrain.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')

In [45]:
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
       'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [51]:
df_fulltrain[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64